# 03 — News Processing Chain

## Objective

Build the news prompt-chaining workflow for the investment research system:

**ingest → preprocess → classify → extract → summarize**

Notebooks `00_data_ingestion.ipynb` and `01_data_preprocessing.ipynb` already complete ingestion and preprocessing. This notebook starts from `news_clean.csv` and performs the remaining three stages:

1. **Classify** each relevant article into a financial-news category.
2. **Extract** structured facts supported by the article.
3. **Summarize** the extracted evidence into a concise research note.

The output is saved as `data/processed/news_research_results.csv` for the downstream Planner and Router agents.

> The category produced here describes the article content. It is **not** the same as Rajni's Router Agent, which decides which downstream specialist agent should handle evidence.


## 1. Setup

This notebook uses an OpenAI-compatible chat-completions client so the team can point it at the chosen LLM provider.

Add the client dependency once:

```bash
uv add openai
```

Add these values to your local `.env`:

```text
LLM_API_KEY=your_key_here
LLM_MODEL=your_model_name

# Optional for another OpenAI-compatible provider.
LLM_BASE_URL=

# Keep small while developing to control API usage.
NEWS_CHAIN_LIMIT=10

# Set true only when intentionally regenerating existing results.
FORCE_NEWS_CHAIN=false
```

Do not commit `.env`.


In [1]:
import ast
import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env", override=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from shared import PATHS, ensure_directories  # noqa: E402

ensure_directories()

NEWS_INPUT = PATHS["processed"] / "news_clean.csv"
NEWS_OUTPUT = PATHS["processed"] / "news_research_results.csv"

LLM_API_KEY = os.getenv("LLM_API_KEY", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "").strip()

NEWS_CHAIN_LIMIT = int(
    os.getenv("NEWS_CHAIN_LIMIT", "10")
)

FORCE_NEWS_CHAIN = (
    os.getenv("FORCE_NEWS_CHAIN", "false")
    .strip()
    .lower()
    in {"1", "true", "yes"}
)

print("Project root:", PROJECT_ROOT)
print("Input:", NEWS_INPUT)
print("Output:", NEWS_OUTPUT)
print("Model configured:", bool(LLM_MODEL))
print("API key loaded:", bool(LLM_API_KEY))
print("Article limit:", NEWS_CHAIN_LIMIT)
print("Force regeneration:", FORCE_NEWS_CHAIN)

if not NEWS_INPUT.exists():
    raise FileNotFoundError(
        "news_clean.csv was not found. Run 01_data_preprocessing.ipynb first."
    )

if not LLM_API_KEY:
    raise ValueError("LLM_API_KEY is missing from .env.")

if not LLM_MODEL:
    raise ValueError("LLM_MODEL is missing from .env.")


Project root: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System
Input: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\news_clean.csv
Output: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\news_research_results.csv
Model configured: True
API key loaded: True
Article limit: 10
Force regeneration: False


## 2. Load Relevance-Filtered News

In [2]:
news = pd.read_csv(NEWS_INPUT)

required_columns = [
    "ticker",
    "published_at",
    "title",
    "source",
    "url",
    "text",
    "relevance_score",
]

missing_columns = [
    column
    for column in required_columns
    if column not in news.columns
]

if missing_columns:
    raise ValueError(
        f"news_clean.csv is missing required columns: {missing_columns}"
    )

news["published_at"] = pd.to_datetime(
    news["published_at"],
    errors="coerce",
    utc=True,
)

news["relevance_score"] = pd.to_numeric(
    news["relevance_score"],
    errors="coerce",
)

news = news.dropna(
    subset=[
        "ticker",
        "published_at",
        "title",
        "text",
    ]
).copy()

news = news[
    news["relevance_score"] >= 1
].copy()

news = (
    news.sort_values(
        ["relevance_score", "published_at"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print("Relevant cleaned articles available:", len(news))

display(
    news[
        [
            "ticker",
            "published_at",
            "title",
            "source",
            "relevance_score",
        ]
    ].head(10)
)


Relevant cleaned articles available: 24


,ticker,published_at,title,source,relevance_score
0,AAPL,2026-09-20 16:37:00+00:00,"Prediction: Even With the $1,999 Price Tag, Ap...",finance.yahoo.com,2
1,AAPL,2026-09-20 16:25:40+00:00,AAPL Looks 17.3% Overvalued on GF Value™ as In...,gurufocus.com,2
2,AAPL,2026-09-20 11:30:29+00:00,Apple Enhances Durability with New A20 Pro Chi...,gurufocus.com,2
3,AAPL,2026-09-19 12:50:00+00:00,John Ternus's First iPhone Launch Prompted a B...,finance.yahoo.com,2
4,AAPL,2026-09-18 18:11:54+00:00,Apple Raised Stakes with its $2000-Priced Fold...,insidermonkey.com,2
5,AAPL,2026-09-18 17:57:20+00:00,AAPL Looks 16.9% Overvalued on GF Value™ Amid ...,gurufocus.com,2
6,AAPL,2026-09-18 16:55:26+00:00,META Looks 21.3% Undervalued on GF Value™ as M...,gurufocus.com,2
7,AAPL,2026-09-18 15:30:05+00:00,Apple Stock Has a New Opportunity Investors Ca...,finance.yahoo.com,2
8,AAPL,2026-09-18 13:30:58+00:00,AAPL Maintained by Evercore ISI Group -- Price...,gurufocus.com,2
9,AAPL,2026-09-18 13:30:00+00:00,"Apple Stock: The $1,999 iPhone Hides A Bigger ...",seekingalpha.com,2


## 3. Financial-News Categories

The classifier assigns each article to exactly one content category.

These categories describe **what the article is about**. They do not decide which agent should handle it.


In [3]:
NEWS_CATEGORIES = [
    "earnings",
    "product_service",
    "management",
    "regulation_legal",
    "merger_acquisition",
    "analyst_investor",
    "macro_market",
    "other",
]

NEWS_CATEGORIES


['earnings',
 'product_service',
 'management',
 'regulation_legal',
 'merger_acquisition',
 'analyst_investor',
 'macro_market',
 'other']

## 4. LLM Client and Structured-Output Helpers

In [4]:
def build_client():
    if LLM_BASE_URL:
        return OpenAI(
            api_key=LLM_API_KEY,
            base_url=LLM_BASE_URL,
        )

    return OpenAI(
        api_key=LLM_API_KEY,
    )


client = build_client()


def parse_json_response(raw_text):
    text = raw_text.strip()

    if text.startswith("```"):
        text = text.strip("`")

        if text.lower().startswith("json"):
            text = text[4:].strip()

    try:
        return json.loads(text)

    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(text)

            if isinstance(parsed, dict):
                return parsed

        except (ValueError, SyntaxError):
            pass

    raise ValueError(
        f"Model response was not valid JSON: {raw_text}"
    )


def call_llm_json(
    system_prompt,
    user_prompt,
    temperature=0,
):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
    )

    content = response.choices[0].message.content

    if not content:
        raise ValueError("The model returned an empty response.")

    return parse_json_response(content)


## 5. Stage 1 — Classify

The first prompt receives only the cleaned article text and returns a single financial-news category plus a short rationale.

Keeping this as its own step makes the required prompt chain explicit and allows later prompts to use the classification result.


In [5]:
def classify_article(article_text):
    categories = ", ".join(NEWS_CATEGORIES)

    system_prompt = """
You classify financial news for an investment research pipeline.

Use only the article provided by the user.
Do not make investment recommendations.
Return valid JSON only.
""".strip()

    user_prompt = f"""
Classify this article into exactly one of these categories:

{categories}

Category definitions:
- earnings: earnings reports, revenue, profit, guidance, or financial results
- product_service: products, services, launches, technology, or operations
- management: executives, boards, leadership, hiring, or departures
- regulation_legal: lawsuits, regulation, investigations, or government action
- merger_acquisition: mergers, acquisitions, divestitures, or major strategic deals
- analyst_investor: analyst ratings, price targets, investor commentary, or capital markets
- macro_market: broader economic, sector, commodity, interest-rate, or market developments
- other: relevant company news that does not fit the categories above

Return exactly this JSON structure:

{{
  "category": "one_category_from_the_list",
  "classification_reason": "one concise sentence"
}}

Article:
{article_text}
""".strip()

    result = call_llm_json(
        system_prompt,
        user_prompt,
    )

    category = result.get("category")

    if category not in NEWS_CATEGORIES:
        raise ValueError(
            f"Invalid category returned by model: {category}"
        )

    return result


## 6. Stage 2 — Extract

The extraction prompt receives both the article and the category from Stage 1.

It converts unstructured news into structured research evidence while being instructed not to invent unsupported facts.


In [6]:
def extract_article_facts(
    article_text,
    category,
):
    system_prompt = """
You extract structured evidence from financial news.

Use only facts explicitly supported by the supplied article.
Do not infer missing facts.
Do not make investment recommendations.
Return valid JSON only.
""".strip()

    user_prompt = f"""
The article was classified as:

{category}

Extract the important research evidence from the article.

Return exactly this JSON structure:

{{
  "event": "one-sentence description of the primary event",
  "key_facts": [
    "fact 1",
    "fact 2"
  ],
  "people": [
    "person name"
  ],
  "organizations": [
    "organization name"
  ],
  "financial_numbers": [
    "financial or quantitative value exactly as stated"
  ]
}}

Use empty lists when a field has no supported information.

Article:
{article_text}
""".strip()

    result = call_llm_json(
        system_prompt,
        user_prompt,
    )

    expected_list_fields = [
        "key_facts",
        "people",
        "organizations",
        "financial_numbers",
    ]

    for field in expected_list_fields:
        value = result.get(field)

        if value is None:
            result[field] = []

        elif not isinstance(value, list):
            result[field] = [str(value)]

    if not result.get("event"):
        result["event"] = ""

    return result


## 7. Stage 3 — Summarize

The final prompt is intentionally grounded in the **structured extraction from Stage 2**, rather than asking the model to start from scratch.

This demonstrates actual prompt chaining:

**classification output → extraction output → summary output**


In [7]:
def summarize_article(
    category,
    extracted_facts,
):
    system_prompt = """
You write concise evidence-grounded financial research summaries.

Use only the structured evidence supplied by the user.
Do not introduce new facts.
Do not make a buy, sell, or hold recommendation.
Return valid JSON only.
""".strip()

    evidence_json = json.dumps(
        extracted_facts,
        ensure_ascii=False,
        indent=2,
    )

    user_prompt = f"""
Article category:

{category}

Structured evidence:

{evidence_json}

Write a concise research summary that:
1. identifies the main event,
2. includes the most important supported facts,
3. briefly explains why the event may matter for company research,
4. avoids unsupported conclusions or investment recommendations.

Return exactly:

{{
  "summary": "2-4 sentence research summary"
}}
""".strip()

    result = call_llm_json(
        system_prompt,
        user_prompt,
    )

    return result


## 8. Full Prompt Chain

In [8]:
def run_news_chain(article):
    classification = classify_article(
        article["text"]
    )

    extraction = extract_article_facts(
        article_text=article["text"],
        category=classification["category"],
    )

    summary_result = summarize_article(
        category=classification["category"],
        extracted_facts=extraction,
    )

    return {
        "ticker": article["ticker"],
        "published_at": article["published_at"],
        "title": article["title"],
        "source": article.get("source"),
        "url": article.get("url"),
        "relevance_score": article.get("relevance_score"),
        "category": classification["category"],
        "classification_reason": classification.get(
            "classification_reason",
            "",
        ),
        "event": extraction.get("event", ""),
        "key_facts": extraction.get("key_facts", []),
        "people": extraction.get("people", []),
        "organizations": extraction.get(
            "organizations",
            [],
        ),
        "financial_numbers": extraction.get(
            "financial_numbers",
            [],
        ),
        "summary": summary_result.get(
            "summary",
            "",
        ),
    }


## 9. Run the Chain

During development, `NEWS_CHAIN_LIMIT` should stay small to control API cost.

Existing results are reused by URL unless `FORCE_NEWS_CHAIN=true`. This also allows the notebook to resume after an interrupted run.


In [9]:
existing_results = pd.DataFrame()

if NEWS_OUTPUT.exists() and not FORCE_NEWS_CHAIN:
    existing_results = pd.read_csv(
        NEWS_OUTPUT
    )

    print(
        "Existing research rows:",
        len(existing_results),
    )


completed_urls = set()

if (
    not existing_results.empty
    and "url" in existing_results.columns
):
    completed_urls = set(
        existing_results["url"]
        .dropna()
        .astype(str)
    )


articles_to_process = news.copy()

if completed_urls and not FORCE_NEWS_CHAIN:
    articles_to_process = articles_to_process[
        ~articles_to_process["url"]
        .astype(str)
        .isin(completed_urls)
    ]


articles_to_process = articles_to_process.head(
    NEWS_CHAIN_LIMIT
)


print(
    "Articles to process this run:",
    len(articles_to_process),
)


Articles to process this run: 10


In [10]:
new_results = []

for _, article in articles_to_process.iterrows():
    print(
        "Processing:",
        article["ticker"],
        "-",
        article["title"][:80],
    )

    try:
        result = run_news_chain(
            article
        )

        new_results.append(
            result
        )

    except Exception as exc:
        print(
            "Failed:",
            article["title"][:80],
        )
        print(
            type(exc).__name__,
            str(exc),
        )


new_results_df = pd.DataFrame(
    new_results
)


if existing_results.empty:
    results = new_results_df.copy()

elif new_results_df.empty:
    results = existing_results.copy()

else:
    results = pd.concat(
        [
            existing_results,
            new_results_df,
        ],
        ignore_index=True,
    )


if not results.empty:
    results = results.drop_duplicates(
        subset=["url"],
        keep="last",
    )

    results.to_csv(
        NEWS_OUTPUT,
        index=False,
    )


print(
    "New successful rows:",
    len(new_results_df),
)

print(
    "Total saved research rows:",
    len(results),
)

if not results.empty:
    display(
        results[
            [
                "ticker",
                "title",
                "category",
                "event",
                "summary",
            ]
        ].tail(10)
    )


Processing: AAPL - Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be 
Processing: AAPL - AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists
Processing: AAPL - Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 
Processing: AAPL - John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. I
Processing: AAPL - Apple Raised Stakes with its $2000-Priced Foldable. But Can it Be the New Growth
Processing: AAPL - AAPL Looks 16.9% Overvalued on GF Value™ Amid Strong Upgrade Cycle Outlook
Processing: AAPL - META Looks 21.3% Undervalued on GF Value™ as Muse App Tops Apple Store
Processing: AAPL - Apple Stock Has a New Opportunity Investors Can’t Afford to Ignore
Processing: AAPL - AAPL Maintained by Evercore ISI Group -- Price Target Raised to $380
Processing: AAPL - Apple Stock: The $1,999 iPhone Hides A Bigger Problem (NASDAQ:AAPL)
New successful rows: 10
Total saved research rows: 10


,ticker,title,category,event,summary
0,AAPL,"Prediction: Even With the $1,999 Price Tag, Ap...",product_service,"Prediction that Apple's foldable iPhone Duo, p...",The main event is a prediction that Apple's fo...
1,AAPL,AAPL Looks 17.3% Overvalued on GF Value™ as In...,analyst_investor,Apple Inc. is reportedly 17.3% overvalued on G...,Apple Inc. is reportedly 17.3% overvalued on G...
2,AAPL,Apple Enhances Durability with New A20 Pro Chi...,product_service,Apple unveils the A20 Pro chip and Ceramic Shi...,"Apple unveiled the A20 Pro chip, Ceramic Shiel..."
3,AAPL,John Ternus's First iPhone Launch Prompted a B...,analyst_investor,Bank of America cuts Apple’s price target afte...,Bank of America cut Apple's price target after...
4,AAPL,Apple Raised Stakes with its $2000-Priced Fold...,product_service,"Apple released its first foldable phone, the D...","Apple released its first foldable phone, the D..."
5,AAPL,AAPL Looks 16.9% Overvalued on GF Value™ Amid ...,analyst_investor,Apple is viewed as 16.9% overvalued on GF Valu...,Apple is viewed as 16.9% overvalued on GF Valu...
6,AAPL,META Looks 21.3% Undervalued on GF Value™ as M...,product_service,Meta Platforms Inc.'s Muse app topped Apple's ...,Meta Platforms Inc.'s Muse app topped Apple's ...
7,AAPL,Apple Stock Has a New Opportunity Investors Ca...,earnings,Apple reported its ninth consecutive earnings ...,Apple reported its ninth consecutive earnings ...
8,AAPL,AAPL Maintained by Evercore ISI Group -- Price...,analyst_investor,Evercore ISI Group maintained an Outperform ra...,Evercore ISI Group maintained an Outperform ra...
9,AAPL,"Apple Stock: The $1,999 iPhone Hides A Bigger ...",analyst_investor,Analyst argues Apple stock is a sell due to a ...,An analyst argues that Apple stock is a sell d...


## 10. Output Quality Checks

In [11]:
if results.empty:
    print(
        "No research results are available yet."
    )

else:
    quality_checks = {
        "rows": len(results),
        "missing_category": int(
            results["category"].isna().sum()
        ),
        "missing_event": int(
            results["event"].isna().sum()
        ),
        "missing_summary": int(
            results["summary"].isna().sum()
        ),
        "missing_url": int(
            results["url"].isna().sum()
        ),
        "duplicate_urls": int(
            results["url"]
            .dropna()
            .duplicated()
            .sum()
        ),
        "invalid_categories": int(
            (~results["category"].isin(
                NEWS_CATEGORIES
            )).sum()
        ),
    }

    display(
        pd.DataFrame(
            [
                {
                    "check": key,
                    "value": value,
                }
                for key, value in quality_checks.items()
            ]
        )
    )

    category_distribution = (
        results["category"]
        .value_counts()
        .rename_axis("category")
        .reset_index(name="count")
    )

    display(
        category_distribution
    )


,check,value
0,rows,10
1,missing_category,0
2,missing_event,0
3,missing_summary,0
4,missing_url,0
5,duplicate_urls,0
6,invalid_categories,0


,category,count
0,analyst_investor,5
1,product_service,4
2,earnings,1


## 11. Manual Sample Review

Automated checks confirm structure, but a small qualitative review is also useful.

For the final report, manually inspect a sample and verify that:

- the category matches the article topic;
- extracted facts are supported by the article;
- the summary reflects the extracted evidence;
- no unsupported financial claim was introduced;
- the source URL remains attached for traceability.


In [12]:
if not results.empty:
    review_columns = [
        "ticker",
        "title",
        "relevance_score",
        "category",
        "classification_reason",
        "event",
        "key_facts",
        "summary",
        "url",
    ]

    display(
        results[
            review_columns
        ].head(10)
    )


,ticker,title,relevance_score,category,classification_reason,event,key_facts,summary,url
0,AAPL,"Prediction: Even With the $1,999 Price Tag, Ap...",2,product_service,Centers on a new Apple product (foldable iPhon...,"Prediction that Apple's foldable iPhone Duo, p...","[The foldable iPhone Duo is priced at $1,999.,...",The main event is a prediction that Apple's fo...,https://finance.yahoo.com/markets/stocks/artic...
1,AAPL,AAPL Looks 17.3% Overvalued on GF Value™ as In...,2,analyst_investor,The article centers on stock valuation and ins...,Apple Inc. is reportedly 17.3% overvalued on G...,[AAPL is reportedly 17.3% overvalued on GF Val...,Apple Inc. is reportedly 17.3% overvalued on G...,https://www.gurufocus.com/news/9089254/aapl-lo...
2,AAPL,Apple Enhances Durability with New A20 Pro Chi...,2,product_service,Discusses a new product launch and hardware fe...,Apple unveils the A20 Pro chip and Ceramic Shi...,"[Launch date: September 20, 2026, New A20 Pro ...","Apple unveiled the A20 Pro chip, Ceramic Shiel...",https://www.gurufocus.com/news/9089235/apple-e...
3,AAPL,John Ternus's First iPhone Launch Prompted a B...,2,analyst_investor,Bank of America cut Apple's price target after...,Bank of America cuts Apple’s price target afte...,[Bank of America cut Apple's price target foll...,Bank of America cut Apple's price target after...,https://finance.yahoo.com/markets/stocks/artic...
4,AAPL,Apple Raised Stakes with its $2000-Priced Fold...,2,product_service,Article centers on Apple's launch of the folda...,"Apple released its first foldable phone, the D...","[The Duo is Apple's first foldable phone., The...","Apple released its first foldable phone, the D...",https://www.insidermonkey.com/news/apple-raise...
5,AAPL,AAPL Looks 16.9% Overvalued on GF Value™ Amid ...,2,analyst_investor,Analyst commentary from Evercore ISI on a stro...,Apple is viewed as 16.9% overvalued on GF Valu...,[GF Value™ indicates Apple (AAPL) is 16.9% ove...,Apple is viewed as 16.9% overvalued on GF Valu...,https://www.gurufocus.com/news/9088386/aapl-lo...
6,AAPL,META Looks 21.3% Undervalued on GF Value™ as M...,2,product_service,News about the launch and performance of Meta'...,Meta Platforms Inc.'s Muse app topped Apple's ...,[Muse app surged to No. 1 on Apple's U.S. App ...,Meta Platforms Inc.'s Muse app topped Apple's ...,https://www.gurufocus.com/news/9088304/meta-lo...
7,AAPL,Apple Stock Has a New Opportunity Investors Ca...,2,earnings,The article focuses on Apple’s earnings beat a...,Apple reported its ninth consecutive earnings ...,[Apple just posted its ninth consecutive earni...,Apple reported its ninth consecutive earnings ...,https://finance.yahoo.com/markets/stocks/artic...
8,AAPL,AAPL Maintained by Evercore ISI Group -- Price...,2,analyst_investor,Analyst rating update and price target for App...,Evercore ISI Group maintained an Outperform ra...,[Evercore ISI Group maintained an Outperform r...,Evercore ISI Group maintained an Outperform ra...,https://www.gurufocus.com/news/9087939/aapl-ma...
9,AAPL,"Apple Stock: The $1,999 iPhone Hides A Bigger ...",2,analyst_investor,The piece centers on investor sentiment and st...,Analyst argues Apple stock is a sell due to a ...,"[The $1,999 iPhone is cited in the article., A...",An analyst argues that Apple stock is a sell d...,https://seekingalpha.com/article/4947822-apple...


## 12. Handoff to Planner and Router

The standardized output of this notebook is:

`data/processed/news_research_results.csv`

### Handoff fields

- `ticker`
- `published_at`
- `title`
- `source`
- `url`
- `relevance_score`
- `category`
- `classification_reason`
- `event`
- `key_facts`
- `people`
- `organizations`
- `financial_numbers`
- `summary`

The downstream Planner and Router can consume these structured research records without needing to repeat ingestion, cleaning, or article-level prompt chaining.

### Responsibility Boundary

This notebook completes the news research pipeline:

**ingest → preprocess → classify → extract → summarize**



## Completion Checklist

- [x] Load relevance-filtered news.
- [x] Define constrained financial-news categories.
- [x] Implement classification stage.
- [x] Implement structured fact-extraction stage.
- [x] Implement evidence-grounded summarization stage.
- [x] Chain the three prompts sequentially.
- [x] Preserve source URLs for traceability.
- [x] Save standardized research records.
- [x] Add automated quality checks.
- [x] Add a manual-review view.
- [x] Define the handoff contract for Planner/Router agents.
